In [0]:
from pyspark.sql import functions as F

from pyspark.sql.types import NumericType

In [0]:
path = "/Volumes/setor_eletrico/ocorrencias_emergenciais/aneel_data/ocorrencias-emergenciais-rede-distribuicao-2026.csv"

df_ocorrencias_emergenciais_rede_distribuicao_2026 = (

    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("quote", '"')
    .option("inferSchema", "true")
    .csv(path)
)

display(
    df_ocorrencias_emergenciais_rede_distribuicao_2026
)

In [0]:

#1. Validação do Dataframe

# O que faz:
# Valida se o dataframe está disponível e possui colunas para que o processo possa ser executado

# Como faz:
# Verifica se o DF existe e se possui ao menos uma coluna existente

# Porque é importante:
# Evita a execução das etapas seguintes sem um dataframe existente

# Pergunta respondida
# O dataframe está pronto para ser analisado?

df = df_ocorrencias_emergenciais_rede_distribuicao_2026

if df is None:
    raise ValueError("O Dataframe não foi definido")

if len(df.columns) == 0:
    raise ValueError("O Dataframe não possui colunas")

In [0]:
#2. Informações Gerais

# O que faz:
# Obtém infos básicas sobre o tamanho e a estrutura do DataFrame

# Como faz:
# Calcula a quantidade de registros e colunas do DataFrame

# Por que é importante:
# Define a dimensão do dataset e fornece a base para os cálculos das métricas

# Pergunta respondida:
# Qual é o tamanho e a estrutura básica do dataset?

total_registros = df.count()
total_colunas = len(df.columns)

In [0]:
#3. Construção das Métricas

# O que faz:
# Define automaticamente as métricas usadas para avaliar a qualidade de cada coluna

# Como faz:
# Percorre todas as colunas do DataFrame e calcula a quantidade de valores NULL, vazios e distintos.

# Por que é importante:
# Permite que o schema profile seja utilizado de forma genérica em diferentes tabelas sem depender de nomes de coluna previamente definidos

# Pergunta respondida:
# Quais características básicas de qualidade devem ser avaliadas em cada coluna?

# A função f.col() converte uma string contendo o nome de uma coluna em um objeto manipulável do tipo Column. Ela é essencial para realizar transformações, cálculos e expressões lógicas diretamente nas colunas de um DataFrame do PySpark.

metricas = []

for campo in df.schema.fields:
    nome_coluna = campo.name

    coluna = F.col(nome_coluna)
    coluna_string = coluna.cast("string")
    
    # O método .extend() adiciona todos os elementos de um objeto iterável ao final de uma lista ou array existente, modificando o objeto original diretamente (in-place).

    # Usando .extend()
    # lista_a = [1, 2, 3]
    # lista_a.extend([4, 5])
    # print(lista_a)  # Saída: [1, 2, 3, 4, 5]

    # Comparação com .append()
    # lista_b = [1, 2, 3]
    # lista_b.append([4, 5])
    # print(lista_b)  # Saída: [1, 2, 3, [4, 5]]

    metricas.extend([
        
        F.sum(
            F.when(coluna.isNull(), 1).otherwise(0)
        ).alias(f"{nome_coluna}__null"),

        F.sum(
            F.when(
                coluna.isNotNull() & (F.trim(coluna_string) == ""), 1).otherwise(0)
        ).alias(f"{nome_coluna}__vazio"),
                    
        F.approx_count_distinct(coluna).alias(f"{nome_coluna}__distintos")
    ])

In [0]:
#4. Execução do profile

# O que faz:
# Executa as métricas definidas anteriormente sobre todos os registros do Dataframe

# Como faz:
# Realiza as agregações necessárias para obter os resultados das métricas de cada coluna

# Porque é importante:
# Centraliza a execução das métricas e permite obter uma visão da qualdiade da tabela

# Pergunta respondida:
# Quais são os rsultados das métricas de qualidade calculadas para esta tabela?

resultado = df.agg(*metricas).collect()[0]

#*metricas (List Unpacking): The asterisk (*) is Python's unpacking operator. It takes the list metricas (which likely contains PySpark aggregation expressions like f.sum('col1'), f.max('col2')) and expands it so each element is passed as a separate argument to the function.

In [0]:
#5. Construção do resultado

# O que faz:
# Transforma os resultados brutos das métricas em indicadores de qualidade mais fáceis de interpretar

# Como faz:
# Calcula a quantidade e o percentual de valores preenchidos, NULL, vazios, não preenchidos e distintos para cada coluna

# Por que é importante:
# Permite comparar o comportamento das colunas utilizando indicadores, absolutos e relativos, independente do tamanho da tabela

# Pergunta respondida:
# Qual é o nível de preenchimento e diversidade de cada coluna?

profile = []

for campo in df.schema.fields:

    nome_coluna = campo.name
    tipo_dado = campo.dataType.simpleString()

    qtd_null = resultado[f"{nome_coluna}__null"] or 0
    qtd_vazio = resultado[f"{nome_coluna}__vazio"] or 0
    qtd_distintos = resultado[f"{nome_coluna}__distintos"] or 0

    qtd_nao_preenchios = qtd_null + qtd_vazio

    qtd_preenchidos = (
        total_registros - qtd_nao_preenchios
    )

    pct_null = (
        qtd_null / total_registros * 100 if total_registros > 0 else 0
    )

    pct_vazio = (
        qtd_vazio / total_registros * 100 if total_registros > 0 else 0
    )

    pct_preenchido = (
        qtd_preenchidos / total_registros * 100 if total_registros > 0 else 0
    )

    pct_distintos = (
        qtd_distintos / total_registros * 100 if total_registros > 0 else 0
    )

    profile.append((
        nome_coluna,
        tipo_dado,
        total_registros,
        qtd_preenchidos,
        qtd_null,
        qtd_vazio,
        qtd_nao_preenchios,
        qtd_distintos,
        round(pct_preenchido, 2),
        round(pct_null, 2),
        round(pct_vazio, 2),
        round(pct_distintos, 2)
    ))

In [0]:
#6. Dataframe do schema profile

# O que faz:
# Cria um dataframe consolidado contendo todas as métricas calculadas para cada coluna

# Como faz:
# Organiza os resultados do profile em uma estrutura tabular com uma linha parada cada coluna

# Por que é importante:
# O schema_profile se torna a principal fonte para as análises, visualizações e classificações realizadas nas etapas seguintes

schema_profile = spark.createDataFrame(
    profile, 
    [   
        "coluna",
        "tipo_dado",
        "total_registros",
        "qtd_preenchidos",
        "qtd_null",
        "qtd_vazio",
        "qtd_nao_preenchidos",
        "qtd_distintos",
        "pct_preenchido",
        "pct_null",
        "pct_vazio",
        "pct_distintos"
    ]
)

In [0]:
display(
    schema_profile.orderBy(F.col("pct_preenchido").asc())
)

In [0]:
schema_profile_preenchimento = (
    schema_profile
    .select(
        "coluna", 
        "pct_preenchido", 
        "pct_null"
    )
    .orderBy(
        F.col("pct_preenchido").asc()
    )
)

schema_profile_stacked = (
    schema_profile_preenchimento
    .selectExpr(
        "coluna",
        "pct_preenchido as percentual",
        "'Preenchido' as tipo"
    )
    .union(
        schema_profile_preenchimento
        .selectExpr(
            "coluna",
            "pct_null as percentual",
            "'Nulo' as tipo"
        )
    )
)

display(schema_profile_stacked)

In [0]:
schema_profile_cardinalidade = (
    schema_profile
    .select(
        "coluna",
        "qtd_distintos",
        "pct_distintos"
    )
    .orderBy(F.col("pct_distintos").asc())
)

display(schema_profile_cardinalidade)

In [0]:
schema_profile_distintos = (
    schema_profile
    .select(
        "coluna",
        "qtd_distintos"
    )
    .orderBy(
        F.col("pct_distintos").desc()
    )
)

display(schema_profile_distintos)

In [0]:
schema_profile_estrutura = (
    schema_profile
    .select(
        "coluna",
        "pct_preenchido",
        "pct_distintos"
    )
    .withColumn(
        "perfil_estrutural",

        F.when(
            (F.col("pct_preenchido") >= 95) & 
            (F.col("pct_distintos") >= 80),
            "Alta completude / Alta cardinalidade"
        )

        .when(
            (F.col("pct_preenchido") >= 95) &
            (F.col("pct_distintos") < 10),
            "Alta completude / Baixa cardinalidade"
        )

        .when(
            (F.col("pct_preenchido") < 50) &
            (F.col("pct_distintos") < 10),
            "Baixa completude / Baixa cardinalidade"
        )

        .otherwise(
            "Comportamento intermediário"
        )
    )
)

display(
    schema_profile_estrutura
    .orderBy(
        F.col("pct_preenchido").asc()
    )
    
)

In [0]:
distribution_profile = (
    schema_profile
    .select(
        "coluna",
        "tipo_dado",
        "total_registros",
        "qtd_preenchidos",
        "qtd_distintos",
        "pct_distintos"
    )
)

display(distribution_profile)

In [0]:
df_string = df.select([F.col(c).cast("string").alias(c) for c in df.columns])

frequencia_profile = (
    df_string
    .unpivot(
        ids=[],
        values=df.columns, 
        variableColumnName="coluna", 
        valueColumnName="valor"
    )
    .where(F.col("valor").isNotNull())
    .groupBy("coluna", "valor")
    .count()
)

display(frequencia_profile.orderBy(
        F.col("count").desc()
    ))